## Exercício de Revisão — Semana 06

Este exercício integra todos os tópicos da semana. Use a base `base_vendas_supermercado.xlsx`.

### Cenário
Você foi contratado para analisar as vendas do supermercado. O arquivo chegou com problemas e você precisa limpá-lo antes de entregar o relatório ao gerente.

### Passos

1. **Carregue** a base e crie uma versão suja:
   - Insira 10 NaN em `'Categoria'` e 8 NaN em `'Quantidade'`
   - Insira 5 valores absurdos em `'Valor Líquido'` (ex: R$ 5000, R$ 0.01)
   - Duplique 7 linhas aleatórias

2. **Diagnóstico**: exiba um relatório com:
   - % de NaN por coluna
   - Número de duplicatas
   - Limites IQR para `'Valor Líquido'` (inferior e superior)

3. **Limpeza completa**:
   - Remova duplicatas
   - Preencha NaN de `'Categoria'` com a moda
   - Preencha NaN de `'Quantidade'` com a mediana
   - Remova outliers de `'Valor Líquido'` pelo método IQR

4. **Transformações**:
   - Crie a coluna `'Mes_Nome'` com o nome do mês (ex: "January")
   - Crie a coluna `'Perfil_Compra'` com `pd.cut()` em 3 faixas de `'Valor Líquido'`
   - Crie a coluna `'Tem_Desconto'` com `np.where()`

5. **Enriquecimento**:
   - Crie uma tabela de metas mensais por loja (defina valores à sua escolha)
   - Faça um `merge` do resumo mensal de vendas com essa tabela de metas
   - Identifique quais lojas atingiram a meta em cada mês

6. **Relatório final**: plote um gráfico mostrando o total de vendas por loja e mês.

In [33]:
import pandas as pd
import re
import numpy as np
import matplotlib.pyplot as plt

print("Ferramentas carregadas com sucesso!")

# Carregar os dados do arquivo CSV
Arquivo = r'C:\Users\miran\OneDrive\Desktop\turma-visualizacao-de-dados\alunos\Isaac_Trenard\semana_06_ANOTACOES\notebooks\base\base_vendas_supermercado.xlsx'

df = pd.read_excel(Arquivo)

print("Dados carregados com sucesso!")

Ferramentas carregadas com sucesso!


c:\Users\miran\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)
c:\Users\miran\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Conditional Formatting extension is not supported and will be removed
  warn(msg)


Dados carregados com sucesso!


In [34]:
print(f'{df.shape[0]} linhas e {df.shape[1]} colunas')

df.info()

250 linhas e 12 colunas
<class 'pandas.DataFrame'>
RangeIndex: 250 entries, 0 to 249
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   Data                250 non-null    datetime64[us]
 1   Loja                250 non-null    str           
 2   Cliente             250 non-null    str           
 3   Categoria           250 non-null    str           
 4   Produto             250 non-null    str           
 5   Quantidade          250 non-null    int64         
 6   Preço Unitário      250 non-null    float64       
 7   Desconto %          250 non-null    float64       
 8   Valor Bruto         250 non-null    float64       
 9   Valor Desconto      250 non-null    float64       
 10  Valor Líquido       250 non-null    float64       
 11  Forma de Pagamento  250 non-null    str           
dtypes: datetime64[us](1), float64(5), int64(1), str(5)
memory usage: 23.6 KB


In [35]:
df.head()

,Data,Loja,Cliente,Categoria,Produto,Quantidade,Preço Unitário,Desconto %,Valor Bruto,Valor Desconto,Valor Líquido,Forma de Pagamento
0,2026-01-04,Loja Shopping,Cliente Final,Laticínios,Queijo Mussarela 500g,4,23.53,0.00,94.12,0.00,94.12,Vale Alimentação
1,2026-01-04,Loja Centro,Clube Fidelidade,Mercearia,Óleo de Soja 900ml,4,6.97,0.00,27.88,0.00,27.88,Vale Alimentação
2,2026-02-27,Loja Shopping,Clube Fidelidade,Padaria,Pão Francês kg,4,15.84,0.03,63.36,1.90,61.46,Cartão de Crédito
3,2026-01-12,Loja Norte,Cliente Final,Laticínios,Leite Integral 1L,6,6.21,0.03,37.26,1.12,36.14,Dinheiro
4,2026-05-05,Loja Norte,Clube Fidelidade,Laticínios,Queijo Mussarela 500g,5,26.63,0.05,133.15,6.66,126.49,Pix


In [37]:
df.describe().round(2)

,Data,Quantidade,Preço Unitário,Desconto %,Valor Bruto,Valor Desconto,Valor Líquido
count,250,250.00,250.00,250.00,250.00,250.00,250.00
mean,2026-03-04 11:25:26.400000,4.04,12.40,0.04,47.58,1.72,45.86
min,2026-01-01 00:00:00,1.00,2.46,0.00,2.95,0.00,2.95
25%,2026-02-01 00:00:00,2.00,6.13,0.00,18.34,0.00,16.99
50%,2026-03-04 00:00:00,4.00,9.20,0.03,34.17,0.86,32.02
75%,2026-04-05 06:00:00,6.00,17.49,0.07,65.06,2.06,63.67
max,2026-05-05 00:00:00,8.00,48.53,0.10,197.36,16.30,197.36
std,NaN,2.33,8.34,0.04,41.26,2.55,39.96


1. **Carregue** a base e crie uma versão suja:
   - Insira 10 NaN em `'Categoria'` e 8 NaN em `'Quantidade'`
   - Insira 5 valores absurdos em `'Valor Líquido'` (ex: R$ 5000, R$ 0.01)
   - Duplique 7 linhas aleatórias

In [57]:
df_sujo = df.copy() #Primeiro criamos um novo dataframe para nao estragar o original
np.random.seed(42) #Embaralhamos os dados para nao ter um padrao

print('Novo dataframe criado com sucesso e embaralhado!, agora vamos sujar os dados!')

nan_setting = { 'Preço Unitário': 20, 'Quantidade': 20, 'Forma de Pagamento': 15, 'Quantidade': 15
               }
df_sujo.loc[251, 'Valor Líquido'] = 5000.00
df_sujo.loc[252, 'Valor Líquido'] = 0.01
df_sujo.loc[253, 'Valor Líquido'] = 5000.00
df_sujo.loc[253, 'Valor Líquido'] = 0.02
df_sujo.loc[253, 'Valor Líquido'] = 5000.00


df_sujo.tail(3)


Novo dataframe criado com sucesso e embaralhado!, agora vamos sujar os dados!


,Data,Loja,Cliente,Categoria,Produto,Quantidade,Preço Unitário,Desconto %,Valor Bruto,Valor Desconto,Valor Líquido,Forma de Pagamento
251,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5000.00,NaN
252,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.01,NaN
253,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5000.00,NaN


In [62]:
for coluna, qtd in nan_setting.items():
    indices = np.random.choice(df_sujo.index, size=qtd, replace=False)
    df_sujo.loc[indices, coluna] = np.nan

duplicatas = df_sujo.sample(n=12, random_state=7)
df_sujo = pd.concat([df_sujo, duplicatas], ignore_index=True)
df_sujo = df_sujo.sample(frac=1, random_state=3).reset_index(drop=True)

print("=== Comparação: Original vs Sujo ===")
print(f"{'Métrica':<30} {'Original':>10} {'Sujo':>10}")
print("-" * 52)
print(f"{'Linhas':<30} {df.shape[0]:>10} {df_sujo.shape[0]:>10}")
print(f"{'Colunas':<30} {df.shape[1]:>10} {df_sujo.shape[1]:>10}")
print(f"{'Total de NaN':<30} {df.isnull().sum().sum():>10} {df_sujo.isnull().sum().sum():>10}")
print(f"{'Duplicatas':<30} {df.duplicated().sum():>10} {df_sujo.duplicated().sum():>10}")

=== Comparação: Original vs Sujo ===
Métrica                          Original       Sujo
----------------------------------------------------
Linhas                                250        277
Colunas                                12         12
Total de NaN                            0        138
Duplicatas                              0         20


In [60]:
for coluna, qtd in nan_setting.items():
    indices = np.random.choice(df_sujo.index, size=qtd, replace=False)
    df_sujo.loc[indices, coluna] = np.nan

duplicatas = df_sujo.sample(n=12, random_state=7)
df_sujo = pd.concat([df_sujo, duplicatas], ignore_index=True)
df_sujo = df_sujo.sample(frac=1, random_state=3).reset_index(drop=True)

print("=== Comparação: Original vs Sujo ===")
print(f"{'Métrica':<30} {'Original':>10} {'Sujo':>10}")
print("-" * 52)
print(f"{'Linhas':<30} {df.shape[0]:>10} {df_sujo.shape[0]:>10}")
print(f"{'Colunas':<30} {df.shape[1]:>10} {df_sujo.shape[1]:>10}")
print(f"{'Total de NaN':<30} {df.isnull().sum().sum():>10} {df_sujo.isnull().sum().sum():>10}")
print(f"{'Duplicatas':<30} {df.duplicated().sum():>10} {df_sujo.duplicated().sum():>10}")

=== Comparação: Original vs Sujo ===
Métrica                          Original       Sujo
----------------------------------------------------
Linhas                                250        265
Colunas                                12         12
Total de NaN                            0         85
Duplicatas                              0         13


In [61]:
print("\n=== Análise de NaN por Coluna no DataFrame Sujo ===")
print(f'colunas com dados comprometidas {df_sujo.isnull().sum()}')
print(f'Total de linhas com NaN {df_sujo.isnull().sum().sum()}')


=== Análise de NaN por Coluna no DataFrame Sujo ===
colunas com dados comprometidas Data                   3
Loja                   3
Cliente                3
Categoria              3
Produto                3
Quantidade            19
Preço Unitário        23
Desconto %             3
Valor Bruto            3
Valor Desconto         3
Valor Líquido          0
Forma de Pagamento    19
dtype: int64
Total de linhas com NaN 85


2. **Diagnóstico**: exiba um relatório com:
   - % de NaN por coluna
   - Número de duplicatas
   - Limites IQR para `'Valor Líquido'` (inferior e superior)


In [63]:
pct_nan = (df_sujo.isnull().sum() / len(df_sujo) * 100).round(1)
pct_nan_filtrado = pct_nan[pct_nan > 0].sort_values(ascending=False)
print("=== Percentual de valores ausentes ===")
for coluna, pct in pct_nan_filtrado.items():
    barras = '█' * int(pct / 2)
    print(f"  {coluna:<25} {pct:5.1f}%  {barras}")

=== Percentual de valores ausentes ===
  Preço Unitário             15.5%  ███████
  Quantidade                 13.0%  ██████
  Forma de Pagamento         12.6%  ██████
  Data                        1.1%  
  Loja                        1.1%  
  Cliente                     1.1%  
  Categoria                   1.1%  
  Produto                     1.1%  
  Desconto %                  1.1%  
  Valor Bruto                 1.1%  
  Valor Desconto              1.1%  


In [64]:
# duplicated() retorna True para a 2ª ocorrência em diante
duplicatas_mask = df_sujo.duplicated()
print(f"Total de duplicatas: {duplicatas_mask.sum()}")
print()

# Ver as linhas duplicadas
linhas_dup = df_sujo[duplicatas_mask].head(5)
print("Primeiras 5 linhas duplicadas:")
print(linhas_dup[['Data', 'Loja', 'Produto', 'Quantidade', 'Valor Líquido']].to_string())
print()


Total de duplicatas: 20

Primeiras 5 linhas duplicadas:
          Data              Loja               Produto  Quantidade  Valor Líquido
25  2026-03-18        Loja Norte    Feijão Carioca 1kg         4.0          32.08
77  2026-01-12        Loja Norte     Leite Integral 1L         6.0          36.14
87  2026-02-04  Loja Bairro Alto  Linguiça Toscana 1kg         NaN          88.24
90  2026-03-23        Loja Norte               Maçã kg         2.0          17.60
116 2026-03-05        Loja Norte       Carne Moída 1kg         2.0          67.50



3. **Limpeza completa**:
   - Remova duplicatas
   - Preencha NaN de `'Categoria'` com a moda
   - Preencha NaN de `'Quantidade'` com a mediana
   - Remova outliers de `'Valor Líquido'` pelo método IQR


4. **Transformações**:
   - Crie a coluna `'Mes_Nome'` com o nome do mês (ex: "January")
   - Crie a coluna `'Perfil_Compra'` com `pd.cut()` em 3 faixas de `'Valor Líquido'`
   - Crie a coluna `'Tem_Desconto'` com `np.where()`


5. **Enriquecimento**:
   - Crie uma tabela de metas mensais por loja (defina valores à sua escolha)
   - Faça um `merge` do resumo mensal de vendas com essa tabela de metas
   - Identifique quais lojas atingiram a meta em cada mês



6. **Relatório final**: plote um gráfico mostrando o total de vendas por loja e mês.